In [1]:
from langchain_core.prompts import (
                                        SystemMessagePromptTemplate,
                                        HumanMessagePromptTemplate,
                                        ChatPromptTemplate
                                        )

from langchain_core.runnables import RunnableLambda

In [2]:
# Load environment variables and configure the Groq chat model.
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
    # other params...
)


# LangChain Expression Language Basics

LangChain Expression Language (LCEL) allows two or more runnables to be chained together into a sequence.

- The output of one runnable's `.invoke()` call becomes the input for the next runnable.
- You can create these chains using the pipe operator `|` or the `.pipe()` method.

## Types of LCEL Chains

- Sequential Chain
- Parallel Chain
- Router Chain
- Chain Runnables
- Custom Chain (Runnable Sequence)

## Sequential LCEL Chain

A Sequential LCEL Chain runs components in order, where the output from one step becomes the input for the next step.

In [3]:
system = SystemMessagePromptTemplate.from_template('You are {level} level bot . You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate.from_messages(messages)

# Sequential Chain Here
chain = template | llm

chain.invoke({"level": "beginner", "topics": "Math", "points": 5})

AIMessage(content='1. Math uses numbers, symbols, and logic to solve problems.  \n2. It includes basic operations like addition, subtraction, multiplication, and division.  \n3. Algebra studies variables and equations to find unknown values.  \n4. Geometry deals with shapes, sizes, and properties of space.  \n5. Calculus and statistics help analyze change and data patterns.', additional_kwargs={'reasoning_content': "Okay, the user asked for a short explanation of math in five points. Let me start by breaking down the key areas. First, numbers and operations are fundamental, like addition and multiplication. Then there's algebra, which deals with variables and equations. Geometry is next, involving shapes and spatial relationships. Calculus is important for understanding change and motion. Lastly, statistics and probability help with data analysis and uncertainty. I need to keep each point concise and avoid technical jargon since the user is a beginner. Let me check if these five areas 

In [4]:
system = SystemMessagePromptTemplate.from_template('You are {level} level bot . You answer in short sentences.')

question = HumanMessagePromptTemplate.from_template('tell me about the {topics} in {points} points')

messages = [system, question]
template = ChatPromptTemplate.from_messages(messages)
# template.invoke({"level": "beginner", "topics": "Math", "points": 5})

# Sequential Chain Here
chain = template | llm

chain.invoke({"level": "beginner", "topics": "Math", "points": 5})

AIMessage(content='1. Math helps solve problems using numbers, shapes, and patterns.  \n2. It includes branches like algebra, geometry, and calculus.  \n3. Used in daily life for budgeting, cooking, and measuring.  \n4. Builds logical thinking and analytical skills.  \n5. Foundational for science, technology, and engineering.', additional_kwargs={'reasoning_content': "Okay, the user wants to know about math in five points. Let me start by breaking down the key areas. First, math is foundational in many fields like science and engineering. That's a good point. Next, it helps with problem-solving and logical thinking. That's another important aspect. Then, there are different branches like algebra, geometry, calculus. That covers the structure of math. Also, math is used in everyday life, like managing money or measuring things. Lastly, it's a universal language that everyone uses, regardless of where they're from. Let me check if these points are clear and concise. Yeah, they cover the 

## Parallel Chain

A Parallel Chain runs multiple runnables at the same time and combines their outputs into a single result.

In [5]:
from langchain_core.runnables import RunnableParallel

In [6]:
template = ChatPromptTemplate.from_template("Give me a Poem in Cricket")

# Sequential Chain Here
chain1 = template | llm

In [7]:
template = ChatPromptTemplate.from_template("Give me a Poem in football")

# Sequential Chain Here
chain2 = template | llm

In [8]:
parallel_chain = RunnableParallel(
    cricket=chain1,
    football=chain2,
)
parallel_chain.invoke({})

{'cricket': AIMessage(content='**Ode to the Cricketing Dance**  \n\nBeneath the blaze of summer’s golden sun,  \nWhere emerald fields and cobalt skies are one,  \nThe crack of willow, the spin of the ball,  \nA symphony of struggle, a dance through it all.  \n\nThe batsman steps, a warrior in white,  \nHis blade a crescent moon, fierce in its fight.  \nThe bowler winds up, a coiled spring of grace,  \nA thunderous release, a yorker’s embrace.  \n\nThe fielders leap, a blur of fervent grace,  \nSprinting to claim the sky-kissed lace.  \nGloves clap tight, a catch that ignites,  \nAs hope and despair in moments collide.  \n\nThe crowd’s breath holds, a hush in the air,  \nThen erupts—a storm of cheers, a tempest laid bare.  \nEach run a verse, each wicket a rhyme,  \nA saga of grit etched in the game’s sweet time.  \n\nThere’s poetry in the spin, the swing, the pace,  \nIn a leg-break that turns on fate’s narrowest space.  \nIn a century’s bloom, a partnership’s slow burn,  \nIn the fina

## Router Chain

A Router Chain directs the input to different runnable paths based on the input type or condition.

In [9]:
template = ChatPromptTemplate.from_template("Give a Happy Message to user")

# Sequential Chain Here
chain1 = template | llm

chain1.invoke({})

AIMessage(content="🌈 Hello there! 🌟 Wishing you a day filled with sunshine, smiles, and little moments that make your heart skip a beat! 🌸 Whether it's a kind word, a cozy cup of tea, or a laugh with loved ones—cherish it all. You’re capable of amazing things, and the world is brighter because you’re in it. Keep shining, and never forget how wonderful you are! ✨💖 #HappyVibesOnly 🌟", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a cheerful greeting. Use emojis to add some visual positivity. Include something about spreading joy and kindness. Maybe mention little things that bring happiness. Add a motivational note about each day being a new opportunity. Keep it short and warm. Let me put it all together in a friendly tone.\n'}, response_metadata={'token_usage': {'completion_tokens': 183, 'prompt_tokens': 14, 'total_tokens': 197, 'completion_time': 0.452911329, 'completion_tokens_de

In [10]:
template = ChatPromptTemplate.from_template("Give me a Sad Message to the user")

# Sequential Chain Here
chain2 = template | llm

chain2.invoke({})

AIMessage(content="Here’s a message crafted to carry the weight of quiet sorrow:  \n\n---  \nThe world keeps spinning, but sometimes it feels like you’re standing still in its shadow—watching the light move forward while you’re left holding the echoes of what once was. The silence isn’t empty; it’s full of all the words that never landed, the laughter that faded too soon, and the love that slipped through your fingers like sand. You don’t have to explain the ache in your chest to anyone. It’s yours, and it’s valid, even when the world tries to tell you otherwise.  \n---  \n\nLet me know if you'd like adjustments—this can be as heavy or subtle as needed.", additional_kwargs={'reasoning_content': 'Okay, the user wants a sad message. Let me think about how to approach this. They might be looking for something heartfelt, maybe to express their own feelings or to share with someone else. I need to make sure the message is genuine and not too over the top. Let me start by considering common 

In [11]:
def route(info):
    if "happy" in info["sentiment"]:
        return chain1.invoke({})
    elif "sad" in info["sentiment"]:
        return chain2.invoke({})

In [12]:
def return_sentiment(info):
    template = ChatPromptTemplate.from_template("Based on the User statement {statement} either give a single word output: happy or sad")
    chain = template | llm
    sentiment = chain.invoke({"statement": info["statement"]}).content.strip().lower()
    return {"sentiment" : sentiment}

In [13]:
statement = "I am feeling great today!"

result_sentiment = return_sentiment({"statement": statement})
full_chain = {"sentiment": RunnableLambda(lambda x: result_sentiment["sentiment"])} | RunnableLambda(route)
full_chain.invoke({})

AIMessage(content="🌈 Hello there! Wishing you a day filled with little joys, big smiles, and moments that make your heart skip a beat! 🌟 Whether it's the sun shining just right, a cozy cup of tea, or a laugh with someone you love—cherish it all. You’re deserving of every happy thing life has to offer. Keep spreading positivity, and remember: good vibes are contagious! 💫✨ Have an absolutely wonderful day! 🌸", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a friendly greeting. Use emojis to add some warmth. Include something about spreading joy and positivity. Maybe mention little things that bring happiness. Add wishes for good vibes and a bright day. Keep it short and cheerful. Let me put that all together in a natural way.\n'}, response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 14, 'total_tokens': 190, 'completion_time': 0.450012894, 'completion_tokens_d

In [14]:
statement = "I am feeling great today!"

def return_sentiment_1(info):
    template = ChatPromptTemplate.from_template("Based on the User statement {statement} either give a single word output: happy or sad")
    chain = template | llm
    sentiment = chain.invoke({"statement": info["statement"]}).content.strip().lower()
    return {"sentiment": sentiment}

def route_1(info):
    if "happy" in info["sentiment"]:
        return chain1
    elif "sad" in info["sentiment"]:
        return chain2

full_chain = {"statement": RunnableLambda(lambda x: statement)} | RunnableLambda(return_sentiment) | RunnableLambda(route_1)
full_chain.invoke({})

AIMessage(content="🌈 Hello there! Wishing you a day filled with little joys, big smiles, and moments that make your heart skip a beat! 🌸 Whether it's the warmth of the sun, a kind word from a friend, or your favorite song playing at just the right time—cherish it all. You’re deserving of happiness, so let’s spread positivity like confetti! 🎉✨ Have a splendid day! 😊", additional_kwargs={'reasoning_content': 'Okay, the user wants a happy message. Let me think about what makes a message truly uplifting. Maybe start with a cheerful greeting. Use some emojis to add a friendly vibe. Include something about spreading joy and positivity. Maybe mention small joys in life. Add a metaphor like sunshine or flowers. Encourage the person to smile and take a moment to appreciate good things. Keep it short and warm. Let me put that all together in a natural way.\n'}, response_metadata={'token_usage': {'completion_tokens': 184, 'prompt_tokens': 14, 'total_tokens': 198, 'completion_time': 0.496130626, '

In [17]:
template1 = ChatPromptTemplate.from_template("Give me a Poem in Cricket")
c1 = template1 | llm

template2 = ChatPromptTemplate.from_template("Give me a Poem in football")
c2 = template2 | llm

routing_prompt = ChatPromptTemplate.from_template("Based on the user query {query} either give a single word output: cricket or football")
c_routing = routing_prompt | llm

query = "I love cricket"

def routing(info):
    if "cricket" in info["sport"]:
        return c1
    elif "football" in info["sport"]:
        return c2
    
full_chain = {"sport" : RunnableLambda(lambda x: c_routing.invoke({"query": query}).content.strip().lower())} | RunnableLambda(routing)
full_chain.invoke({})

AIMessage(content="**Ode to the Cricket's Dance**  \n\nOn emerald fields where legends play,  \nWhere echoes of the past hold sway,  \nThe bat, a willow’s whispered might,  \nMeets ball in a symphony of flight.  \n\nThe bowler leans, a coiled refrain,  \nHis fingers spin a web of pain.  \nThe batsman stands, a sentinel,  \nTime freezes—then, the clash of steel.  \n\nThe crowd’s breath held in suspenseful hush,  \nAs skies ignite with sixes’ rush.  \nA fielder leaps, a graceful arc,  \nTo claim the sky’s elusive spark.  \n\nThe scoreboard hums its silent song,  \nOf triumphs won and dreams undone.  \nIn victories, cheers like thunder roar,  \nIn defeat, a quiet, fiery roar.  \n\nFrom twilight’s blush to stars’ embrace,  \nThe game weaves tales of time and space.  \nA dance of shadows, sweat, and light,  \nWhere hearts and nations find their fight.  \n\nOh, cricket! More than just a name—  \nA tapestry of joy and flame.  \nA circle where the world feels small,  \nAnd every boundary moves

In [29]:
template1 = ChatPromptTemplate.from_template("Give me a Poem in Cricket")
c1 = template1 | llm

template2 = ChatPromptTemplate.from_template("Give me a Poem in football")
c2 = template2 | llm

routing_prompt = ChatPromptTemplate.from_template("Based on the user query {query} either give a single word output: cricket or football")
c_routing = routing_prompt | llm

query = "I love cricket"

def routing(info):
    if "cricket" in info["sport"]:
        return c1
    elif "football" in info["sport"]:
        return c2
    
full_chain = {"sport" : RunnableLambda(lambda x: c_routing)} | RunnableLambda(routing)
full_chain

{
  sport: RunnableLambda(lambda x: c_routing)
}
| RunnableLambda(routing)

## Chain Runnables

Chain Runnables are reusable runnable components that can be combined to build larger LCEL workflows.

- `RunnableLambda` runs a normal Python function as a runnable.
- `RunnablePassthrough` passes the input forward without changing it.

In [21]:
# RunnablePassthrough

from langchain_core.runnables import RunnablePassthrough, RunnableLambda

In [22]:
def add_prefix(message: str) -> str:
    message = "Prefix: " + message
    return message

def add_suffix(message: str) -> str:
    message = message + " :Suffix"
    return message

In [23]:
template = ChatPromptTemplate.from_template("Process the following message and return one word mentioning whether we need to add a prefix or suffix:\n\n{message}")

chain = template | llm
chain
# chain.invoke({"message": "This is a message add as a suffix"})

ChatPromptTemplate(input_variables=['message'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['message'], input_types={}, partial_variables={}, template='Process the following message and return one word mentioning whether we need to add a prefix or suffix:\n\n{message}'), additional_kwargs={})])
| ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000205F5456E10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000205F550BCD0>, model_name='qwen/qwen3-32b', temperature=1e-08, reasoning_format='parsed', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [24]:
def decide(info):
    decision = info["decision"].content.strip().lower()
    message = info["message"]
    if "suffix" in decision:
        return add_suffix(message)
    return add_prefix(message)

In [ ]:
statement = "This is a message add as a suffix"
chain = (
    {
        "message": RunnablePassthrough(),
        "decision": chain,
    }
    | RunnableLambda(decide)
)
chain.invoke(statement)

'This is a message add as a suffix :Suffix'

## Custom Chain (Runnable Sequence)

A Custom Chain uses a runnable sequence to connect multiple custom steps into a flexible workflow.

In [30]:
from langchain_core.runnables import chain

In [38]:
def custom_chain():
    return {
        'fact': c1.invoke({}),
        'poem': c2.invoke({}),
    }

output = custom_chain()
print(output['fact'])
print('\n\n')
print(output['poem'])

content="**Ode to the Cricket's Dance**  \n\nOn emerald fields where legends play,  \nWhere echoes of the past hold sway,  \nThe bat, a willow’s whispered might,  \nMeets ball in a symphony of flight.  \n\nThe bowler leans, a coiled refrain,  \nHis fingers grip the game’s tight chain.  \nThe batsman stands, a sentinel,  \nTime slows—a breath, a fleeting spell.  \n\nThe pitch, a stage for fate’s design,  \nWhere hopes are spun, and dreams entwined.  \nThe ball, a rebel, curves and spins,  \nA dancer in the sky’s vast grin.  \n\nThe crowd’s roar swells, a storm unleashed,  \nAs boundaries tremble, trembling, breached.  \nA six—a sky of shattered sound!  \nOr a slip, a gasp, a catch unbound.  \n\nThe wicket falls, a silent sigh,  \nYet triumph’s grin or tears deny  \nThe truth that cricket’s heart imparts:  \nBoth victory and loss impart.  \n\nFor in this game, where light and shade  \nAre threads that time and passion made,  \nWe find the soul of toil and grace—  \nA timeless dance, this